<sub>**The Annotated ML Handbook** · Part I: Python</sub>

# Chapter 7: The Pythonic toolkit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-1-python/07-python-toolkit.ipynb)

Companion code for the chapter. Every example from the book is here, in order, ready to run. The line-by-line explanations, definitions and discussion are in the book.

Run the setup cell first, then run the cells from top to bottom.

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

## 7.1 collections: smarter containers · *Intermediate*

**Counter, defaultdict, namedtuple, deque**

In [2]:
from collections import Counter, defaultdict, namedtuple, deque

labels = ["cat", "dog", "cat", "bird", "cat", "dog"]
counts = Counter(labels)
print(counts, counts.most_common(1))
print(counts["fish"])

by_group = defaultdict(list)
for name, group in [("ana", "A"), ("bo", "B"), ("cy", "A")]:
    by_group[group].append(name)
print(dict(by_group))

Point = namedtuple("Point", ["x", "y"])
p = Point(2, 5)
print(p.x, p[1], p)

window = deque(maxlen=3)
for loss in [0.9, 0.7, 0.6, 0.55, 0.54]:
    window.append(loss)
print(list(window), sum(window) / len(window))

Counter({'cat': 3, 'dog': 2, 'bird': 1}) [('cat', 3)]
0
{'A': ['ana', 'cy'], 'B': ['bo']}
2 5 Point(x=2, y=5)
[0.6, 0.55, 0.54] 0.5633333333333334


## 7.2 itertools: loops without the loops · *Intermediate*

**Grids, combinations and chunks**

In [3]:
from itertools import product, combinations, permutations, chain, islice, accumulate

grid = {"lr": [0.1, 0.01], "depth": [3, 5]}
for lr, depth in product(grid["lr"], grid["depth"]):
    print(f"lr={lr}, depth={depth}")
print(list(combinations(["A", "B", "C"], 2)))
print(len(list(permutations(range(4)))))
print(list(chain([1, 2], (3, 4), {5})))
print(list(islice(range(100), 5, 10)))
print(list(accumulate([1, 2, 3, 4])))

lr=0.1, depth=3
lr=0.1, depth=5
lr=0.01, depth=3
lr=0.01, depth=5
[('A', 'B'), ('A', 'C'), ('B', 'C')]
24
[1, 2, 3, 4, 5]
[5, 6, 7, 8, 9]
[1, 3, 6, 10]


## 7.3 functools: partial and reduce · *Advanced*

**Pre-filling arguments**

In [4]:
from functools import partial, reduce

def scale(x, factor, offset=0):
    return x * factor + offset

to_percent = partial(scale, factor=100)
print(to_percent(0.257), to_percent(0.5, offset=1))
print(reduce(lambda a, b: a * b, [1, 2, 3, 4, 5]))

25.7 51.0
120


## 7.4 Context managers · *Intermediate*

**A timer you can wrap around any code**

In [5]:
import time
from contextlib import contextmanager

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.4f} s")

with timer("sum of squares"):
    total = sum(i * i for i in range(300_000))

sum of squares: 0.0212 s


## 7.5 Randomness and reproducibility · *Beginner*

**Seeded random numbers**

In [6]:
import random

random.seed(42)
print(random.random(), random.randint(1, 6))
items = list(range(10))
random.shuffle(items)
print(items, random.choice(items), random.sample(items, 3))
random.seed(42)
print(random.random())

0.6394267984578837 1
[7, 6, 2, 9, 0, 5, 1, 8, 3, 4] 7 [7, 6, 9]
0.6394267984578837


## 7.6 Dates and times · *Beginner*

**datetime basics**

In [7]:
from datetime import datetime, timedelta, date

d = datetime(2026, 3, 14, 9, 30)
print(d.isoformat(), d.strftime("%d %b %Y"))
print(d + timedelta(days=30))
parsed = datetime.strptime("2026-01-05", "%Y-%m-%d")
print((d - parsed).days, date(2026, 3, 14).weekday())

2026-03-14T09:30:00 14 Mar 2026
2026-04-13 09:30:00
68 5


## 7.7 Regular expressions · *Intermediate*

**Extracting structured values from messy text**

In [8]:
import re

text = "Sample S-104 measured 12.5 mg on 2026-02-01; sample S-221 measured 7 mg."
ids = re.findall(r"S-\d+", text)
doses = re.findall(r"(\d+(?:\.\d+)?) mg", text)
date_m = re.search(r"(\d{4})-(\d{2})-(\d{2})", text)
print(ids, [float(x) for x in doses], date_m.groups())
print(re.sub(r"\s+", " ", "too    many   spaces"))

['S-104', 'S-221'] [12.5, 7.0] ('2026', '02', '01')
too many spaces


## 7.8 Measuring speed · *Intermediate*

**Timing small snippets**

In [9]:
import timeit

loop_time = timeit.timeit("s = 0\nfor i in range(1000): s += i", number=2000)
builtin_time = timeit.timeit("sum(range(1000))", number=2000)
print(f"loop {loop_time:.3f}s vs sum() {builtin_time:.3f}s -> {loop_time / builtin_time:.1f}x faster")

loop 0.072s vs sum() 0.025s -> 2.9x faster


## Try it yourself

1. Use `Counter` to find the three most common words in a paragraph (lowercase it and split on whitespace first).
2. Use `itertools.product` to list all 12 combinations of `C in [0.1, 1, 10]` and `kernel in ["linear", "rbf", "poly", "sigmoid"]`.
3. Write a regex that extracts all e-mail addresses from a string.

Worked solutions: [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 7*.